# DATA 200 – Applied Statistical Analysis
## Predicting Student Academic Performance Using Statistical and Predictive Modeling

**Team United**
- Samrat Jung Shahi
- Saksham Smith Sunar
- Rishav Yadav

This Google Colab notebook uses the **UCI Student Performance dataset (Portuguese course)** and generates **12 different presentation-ready visualizations** plus statistical analysis and regression modeling.

**Target:** `G3` (final grade, 0–20).

All figures are automatically saved as **300-DPI PNG files** in `data200_visuals/`.

## 1. Setup

In [ ]:
import os, io, zipfile, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
OUTPUT_DIR='data200_visuals'
os.makedirs(OUTPUT_DIR,exist_ok=True)

def save_chart(filename):
    path=f'{OUTPUT_DIR}/{filename}'
    plt.tight_layout()
    plt.savefig(path,dpi=300,bbox_inches='tight')
    plt.show()
    print('Saved:',path)


## 2. Download and load the official UCI dataset

In [ ]:
UCI_ZIP_URL='https://archive.ics.uci.edu/static/public/320/student+performance.zip'
r=requests.get(UCI_ZIP_URL,timeout=60)
r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    por_file=next(name for name in z.namelist() if name.endswith('student-por.csv'))
    df=pd.read_csv(z.open(por_file),sep=';')
print('Dataset shape:',df.shape)
display(df.head())


## 3. Data quality checks

In [ ]:
print('Rows:',df.shape[0])
print('Columns:',df.shape[1])
print('Missing values:',int(df.isna().sum().sum()))
print('Exact duplicates:',int(df.duplicated().sum()))

df_clean=df.drop_duplicates().copy()
display(df_clean.describe().T.round(3))


## 4. Outlier check (IQR rule)
Outliers are **identified, not automatically deleted**.

In [ ]:
def iqr_summary(data,cols):
    rows=[]
    for col in cols:
        q1=data[col].quantile(.25); q3=data[col].quantile(.75); iqr=q3-q1
        lo=q1-1.5*iqr; hi=q3+1.5*iqr
        mask=(data[col]<lo)|(data[col]>hi)
        rows.append([col,q1,q3,lo,hi,int(mask.sum())])
    return pd.DataFrame(rows,columns=['Variable','Q1','Q3','Lower Bound','Upper Bound','Outlier Count'])

outliers=iqr_summary(df_clean,['age','studytime','failures','absences','G1','G2','G3'])
display(outliers)


# 5. Twelve visualizations

### Visual 1 – Distribution of final grades

In [ ]:
plt.figure(figsize=(10,5.6))
counts=df_clean['G3'].value_counts().sort_index()
plt.bar(counts.index,counts.values)
plt.title('Distribution of Final Grades (G3)',fontsize=15,weight='bold')
plt.xlabel('Final Grade (0–20)'); plt.ylabel('Number of Students'); plt.xticks(range(0,21))
save_chart('01_final_grade_distribution.png')


### Visual 2 – Study time and final grade

In [ ]:
study_labels={1:'<2 hours',2:'2–5 hours',3:'5–10 hours',4:'>10 hours'}
means=df_clean.groupby('studytime')['G3'].mean().reindex([1,2,3,4])
plt.figure(figsize=(9,5.6))
plt.bar([study_labels[i] for i in means.index],means.values)
plt.title('Average Final Grade by Weekly Study Time',fontsize=15,weight='bold')
plt.xlabel('Weekly Study Time'); plt.ylabel('Mean Final Grade (G3)'); plt.ylim(0,20)
for i,v in enumerate(means.values): plt.text(i,v+.25,f'{v:.2f}',ha='center')
save_chart('02_study_time_vs_grade.png')


### Visual 3 – Absences and final grade

In [ ]:
grp=df_clean.groupby('absences')['G3'].agg(['mean','count']).reset_index()
plt.figure(figsize=(9,5.6))
plt.scatter(grp['absences'],grp['mean'],s=np.maximum(20,grp['count']*2.2),alpha=.65)
plt.title('Mean Final Grade by Number of Absences',fontsize=15,weight='bold')
plt.xlabel('Absences'); plt.ylabel('Mean Final Grade (G3)'); plt.ylim(0,20)
save_chart('03_absences_vs_grade.png')


### Visual 4 – Previous grade (G2) and final grade

In [ ]:
grp=df_clean.groupby('G2')['G3'].mean().reset_index()
plt.figure(figsize=(9,5.6))
plt.plot(grp['G2'],grp['G3'],marker='o')
plt.plot([0,20],[0,20],linestyle='--')
plt.title('Previous Grade (G2) vs Final Grade (G3)',fontsize=15,weight='bold')
plt.xlabel('Second-Period Grade (G2)'); plt.ylabel('Mean Final Grade (G3)'); plt.xlim(0,20); plt.ylim(0,20)
save_chart('04_G2_vs_G3.png')


### Visual 5 – Correlation heatmap

In [ ]:
num_cols=['age','Medu','Fedu','traveltime','studytime','failures','famrel','freetime','goout','Dalc','Walc','health','absences','G1','G2','G3']
corr=df_clean[num_cols].corr()
plt.figure(figsize=(11,9))
im=plt.imshow(corr.values,aspect='auto',vmin=-1,vmax=1)
plt.xticks(range(len(num_cols)),num_cols,rotation=60,ha='right')
plt.yticks(range(len(num_cols)),num_cols)
plt.title('Correlation Heatmap of Numerical Variables',fontsize=15,weight='bold')
plt.colorbar(im,fraction=.046,pad=.04,label='Correlation')
save_chart('05_correlation_heatmap.png')

display(corr['G3'].drop('G3').sort_values(key=np.abs,ascending=False).to_frame('Correlation with G3').round(3))


### Visual 6 – Family support and final grade

In [ ]:
grouped=[df_clean.loc[df_clean['famsup']==g,'G3'] for g in ['no','yes']]
plt.figure(figsize=(8,5.6))
plt.boxplot(grouped,labels=['No','Yes'])
plt.title('Family Educational Support and Final Grade',fontsize=15,weight='bold')
plt.xlabel('Family Educational Support'); plt.ylabel('Final Grade (G3)'); plt.ylim(0,20)
save_chart('06_family_support_grade.png')


### Visual 7 – School comparison

In [ ]:
means=df_clean.groupby('school')['G3'].mean().sort_values()
labels=['Mousinho da Silveira (MS)' if x=='MS' else 'Gabriel Pereira (GP)' for x in means.index]
plt.figure(figsize=(9,5.2))
plt.barh(labels,means.values)
plt.title('Average Final Grade by School',fontsize=15,weight='bold')
plt.xlabel('Mean Final Grade (G3)'); plt.xlim(0,20)
for i,v in enumerate(means.values): plt.text(v+.15,i,f'{v:.2f}',va='center')
save_chart('07_school_comparison.png')


### Visual 8 – Higher-education intention

In [ ]:
means=df_clean.groupby('higher')['G3'].mean().reindex(['no','yes'])
x=np.arange(len(means))
plt.figure(figsize=(8,5.6))
plt.vlines(x,0,means.values)
plt.scatter(x,means.values,s=120)
plt.xticks(x,['No','Yes'])
plt.title('Higher-Education Intention and Final Grade',fontsize=15,weight='bold')
plt.xlabel('Plans to Pursue Higher Education'); plt.ylabel('Mean Final Grade (G3)'); plt.ylim(0,20)
for i,v in enumerate(means.values): plt.text(i,v+.35,f'{v:.2f}',ha='center')
save_chart('08_higher_education_intent.png')


### Visual 9 – Previous class failures

In [ ]:
g=df_clean.groupby('failures')['G3'].agg(['mean','count']).reset_index()
plt.figure(figsize=(9,5.6))
plt.scatter(g['failures'],g['mean'],s=g['count']*1.2,alpha=.65)
plt.plot(g['failures'],g['mean'],linestyle='--')
plt.title('Previous Failures and Final Grade',fontsize=15,weight='bold')
plt.xlabel('Number of Previous Class Failures'); plt.ylabel('Mean Final Grade (G3)'); plt.xticks(g['failures']); plt.ylim(0,20)
save_chart('09_failures_vs_grade.png')


### Visual 10 – Mother's education level

In [ ]:
means=df_clean.groupby('Medu')['G3'].mean()
plt.figure(figsize=(9,5.6))
plt.plot(means.index,means.values,marker='o')
plt.title("Mother's Education Level and Final Grade",fontsize=15,weight='bold')
plt.xlabel("Mother's Education Level (0–4)"); plt.ylabel('Mean Final Grade (G3)'); plt.xticks(means.index); plt.ylim(0,20)
save_chart('10_mother_education_vs_grade.png')


### Visual 11 – Grade bands by home internet access
For visualization only, scores are grouped as `<10`, `10–14`, and `15+`.

In [ ]:
tmp=df_clean[['internet','G3']].copy()
tmp['band']=pd.cut(tmp['G3'],bins=[-1,9,14,20],labels=['Below 10','10–14','15+'])
pct=pd.crosstab(tmp['internet'],tmp['band'],normalize='index')*100
pct=pct.reindex(['no','yes'])
x=np.arange(len(pct))
bottom=np.zeros(len(pct))
plt.figure(figsize=(9,5.6))
for band in ['Below 10','10–14','15+']:
    vals=pct[band].values
    plt.bar(x,vals,bottom=bottom,label=band)
    bottom+=vals
plt.xticks(x,['No Internet','Internet'])
plt.ylabel('Students (%)'); plt.ylim(0,100)
plt.title('Grade Bands by Home Internet Access',fontsize=15,weight='bold')
plt.legend()
save_chart('11_internet_grade_bands.png')


### Visual 12 – Weekend alcohol consumption

In [ ]:
means=df_clean.groupby('Walc')['G3'].mean()
plt.figure(figsize=(9,5.6))
plt.plot(means.index,means.values,marker='o')
plt.title('Weekend Alcohol Consumption and Final Grade',fontsize=15,weight='bold')
plt.xlabel('Weekend Alcohol Level (1=Very Low, 5=Very High)'); plt.ylabel('Mean Final Grade (G3)'); plt.xticks(means.index); plt.ylim(0,20)
save_chart('12_weekend_alcohol_vs_grade.png')


# 6. Hypothesis testing

In [ ]:
alpha=.05
results=[]

r,p=stats.pearsonr(df_clean['studytime'],df_clean['G3']); results.append(['Study time vs G3','Pearson correlation',r,p,'Reject H0' if p<alpha else 'Fail to reject H0'])
r,p=stats.pearsonr(df_clean['absences'],df_clean['G3']); results.append(['Absences vs G3','Pearson correlation',r,p,'Reject H0' if p<alpha else 'Fail to reject H0'])
groups=[g['G3'].values for _,g in df_clean.groupby('studytime')]; f,p=stats.f_oneway(*groups); results.append(['G3 across study-time groups','One-way ANOVA',f,p,'Reject H0' if p<alpha else 'Fail to reject H0'])
y=df_clean.loc[df_clean['famsup']=='yes','G3']; n=df_clean.loc[df_clean['famsup']=='no','G3']; t,p=stats.ttest_ind(y,n,equal_var=False); results.append(['Family support vs G3','Welch t-test',t,p,'Reject H0' if p<alpha else 'Fail to reject H0'])

hypothesis_table=pd.DataFrame(results,columns=['Test','Method','Statistic','P-value','Decision'])
display(hypothesis_table.round({'Statistic':4,'P-value':6}))


# 7. Multiple Linear Regression

In [ ]:
model_a_numeric=['studytime','failures','absences','Medu','Fedu','traveltime','famrel','freetime','goout','Dalc','Walc','health']
model_a_categorical=['famsup','schoolsup','activities','higher','internet','paid','school','sex','address']
model_b_numeric=model_a_numeric+['G1','G2']
model_b_categorical=model_a_categorical

def formula(target,num,cat): return target+' ~ '+' + '.join(num+[f'C({c})' for c in cat])
ols_a=smf.ols(formula('G3',model_a_numeric,model_a_categorical),data=df_clean).fit()
ols_b=smf.ols(formula('G3',model_b_numeric,model_b_categorical),data=df_clean).fit()
print('MODEL A – WITHOUT G1/G2')
print(ols_a.summary())
print('\nMODEL B – WITH G1/G2')
print(ols_b.summary())


# 8. Train/test model evaluation

In [ ]:
def evaluate(num,cat,name):
    X=df_clean[num+cat].copy(); y=df_clean['G3'].copy()
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42)
    prep=ColumnTransformer([('num','passthrough',num),('cat',OneHotEncoder(drop='first',handle_unknown='ignore'),cat)])
    pipe=Pipeline([('prep',prep),('model',LinearRegression())])
    pipe.fit(Xtr,ytr); pred=pipe.predict(Xte)
    r2=r2_score(yte,pred); mae=mean_absolute_error(yte,pred); rmse=np.sqrt(mean_squared_error(yte,pred))
    p=pipe.named_steps['prep'].transform(Xte).shape[1]; n=len(yte); adj=1-(1-r2)*(n-1)/(n-p-1)
    return pipe,yte,pred,{'Model':name,'R²':r2,'Adjusted R²':adj,'MAE':mae,'RMSE':rmse}

pa,ya,preda,ma=evaluate(model_a_numeric,model_a_categorical,'Model A – Without G1/G2')
pb,yb,predb,mb=evaluate(model_b_numeric,model_b_categorical,'Model B – With G1/G2')
metrics=pd.DataFrame([ma,mb])
display(metrics.round(4))


# 9. Export all 12 visuals

In [ ]:
import shutil
zip_path=shutil.make_archive('DATA200_12_visuals','zip',OUTPUT_DIR)
print('Created:',zip_path)
print('Files:')
for f in sorted(os.listdir(OUTPUT_DIR)): print(' -',f)
try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass


# 10. Best visuals for the final slides
Recommended presentation choices:
- **Slide: Dataset/EDA:** Visual 1 + Visual 2
- **Slide: Relationships:** Visual 4 + Visual 5
- **Slide: Important factors:** Visual 8 + Visual 9
- **Slide: Additional behavioral findings:** Visual 10 + Visual 12

Do not place all 12 graphs in the 10-minute presentation. Use the rest in the report, appendix, or Streamlit application.